# Lichess extract (free, on Google Colab)

Streams one month of the Lichess open database (CC0) on Google's computer, keeps only players who play both bullet and slow games, and saves **one small anonymised file** to your Google Drive (`compression-learner-data/lichess/`).

- Nothing is downloaded to your own computer. The raw dump is streamed and never stored, and the intermediate files live on Colab's temporary computer, which is wiped when the session ends.
- Player names become anonymous IDs made with your secret key. A safety check stops before saving if any name is left.
- **How to run:** Runtime → Run all. You'll be asked for the secret key (any long random text; use the same one every time) and to allow Drive access. Keep this tab open while it runs.
- **First time:** keep `MAX_GAMES = 2_000_000` (about 10–20 min) to check everything works. Then set `MAX_GAMES = 0` for the whole month (a few hours).

Code: https://github.com/saurabh9729/compression-learner-data (`pipelines/lichess/`)

In [ ]:
# Settings
MONTH = "2018-06"        # Lichess month (clock times exist from April 2017)
MAX_GAMES = 2_000_000    # 0 = whole month; same cut is used in both passes
MIN_BULLET = 30          # player needs at least this many bullet games ...
MIN_SLOW = 15            # ... and this many rapid + classical games
N_PLAYERS = 300          # random sample of eligible players

In [ ]:
# Install tools and fetch the pipeline code from the public repo
!apt-get -qq install -y zstd > /dev/null
!pip -q install chess pyarrow
import os, getpass
BASE = "https://raw.githubusercontent.com/saurabh9729/compression-learner-data/main/pipelines/lichess"
os.makedirs("/content/work", exist_ok=True); os.chdir("/content/work")
for f in ["filter_lichess.py", "select_players.py", "lichess_pipeline.py", "check_no_names.py"]:
    !curl -sSfLO {BASE}/{f}
!ls -l *.py

In [ ]:
# Secret key for the anonymous IDs (not saved anywhere; type the same text each run)
os.environ["LICHESS_ID_SALT"] = getpass.getpass("Secret key: ")
assert len(os.environ["LICHESS_ID_SALT"]) >= 12, "use a longer key"
URL = f"https://database.lichess.org/standard/lichess_db_standard_rated_{MONTH}.pgn.zst"
print("source:", URL)

In [ ]:
# Pass 1: count games per player (streamed; nothing stored but the counts)
!curl -sL {URL} | zstdcat 2>/dev/null | python filter_lichess.py --count counts.csv --max-games {MAX_GAMES}
!wc -l < counts.csv

In [ ]:
# Pick players who play both fast and slow games
!python select_players.py counts.csv --min-bullet {MIN_BULLET} --min-slow {MIN_SLOW} --n {N_PLAYERS} --out players.txt

In [ ]:
# Pass 2: keep only their games (with clock times)
!curl -sL {URL} | zstdcat 2>/dev/null | python filter_lichess.py --extract players.txt --max-games {MAX_GAMES} > filtered.pgn
!ls -lh filtered.pgn

In [ ]:
# Per-move measures with anonymous IDs, then the safety check
OUT = f"moves_{MONTH}" + ("" if MAX_GAMES == 0 else f"_first{MAX_GAMES//1_000_000}M")
!python lichess_pipeline.py filtered.pgn --players players.txt --out {OUT}.parquet
!python check_no_names.py players.txt {OUT}.parquet {OUT}_players.csv

In [ ]:
# Save only the two anonymised files to Google Drive (runs only if the safety check passed)
import subprocess, shutil
ok = subprocess.run(["python", "check_no_names.py", "players.txt", f"{OUT}.parquet", f"{OUT}_players.csv"]).returncode == 0
assert ok, "safety check failed: nothing saved"
from google.colab import drive
drive.mount("/content/drive")
dest = "/content/drive/MyDrive/compression-learner-data/lichess"
os.makedirs(dest, exist_ok=True)
for f in [f"{OUT}.parquet", f"{OUT}_players.csv"]:
    shutil.copy(f, dest)
print("saved to Drive:", dest, os.listdir(dest))